In [ ]:
import mlflow
import pandas as pd
from dotenv import load_dotenv
from mlflow import MlflowClient
from mlflow.deployments import get_deploy_client

load_dotenv("../config/.env")
mlflow.set_registry_uri("databricks-uc")

#### Update Model Endpoint

In [ ]:
client_mlflow = MlflowClient()
client_databricks = get_deploy_client("databricks")

In [ ]:
model_version = client_mlflow.get_model_version_by_alias(
    "betsim.models.xgboost_optuna",
    "champion",
)
version = model_version.version

In [ ]:
endpoint = client_databricks.update_endpoint_config(
    endpoint="betsim",
    config={
        "served_entities": [
            {
                "entity_name": "betsim.models.xgboost_optuna",
                "entity_version": version,
                "workload_size": "Small",
                "workload_type": "CPU",
                "scale_to_zero_enabled": True,
            },
        ],
    },
)

In [ ]:
ep = client_databricks.get_endpoint("betsim")
ep["state"]["config_update"]

#### Evaluate Business Value

In [ ]:
def predict_jleague(df: pd.DataFrame):
    data = df.to_dict(orient="split")

    response = client_databricks.predict(
        inputs={"dataframe_split": data},
        endpoint="betsim",
    )

    return pd.DataFrame(response["predictions"])


def get_rob(prediction, y_true, risk, stake=200, odds=1.8):
    prediction["correct"] = prediction["y_hat"] == y_true.values

    match risk:
        case "taker":
            bet = prediction.query("not 0.4 < pi_hat < 0.6")
        case "neutral":
            bet = prediction.query("not 0.3 < pi_hat < 0.7")
        case "averse":
            bet = prediction.query("not 0.2 < pi_hat < 0.8")
        case _:
            bet = prediction.query("is_bet")

    total = len(bet)
    correct = bet["correct"].sum()
    incorrect = total - correct
    pnl = correct * odds - incorrect
    rob = odds * correct / total - 1

    return pnl * stake, rob

In [ ]:
df = pd.read_parquet("../data/processed/jleague_dev.parquet")
df.query("season != 2262", inplace=True)
df.sort_values("gid", inplace=True, ignore_index=True)

df["date"] = df["gdt"].str[:10]

df_test = df.query("season == season.max() - 1")
df_prod = df.query("season == season.max()")

In [ ]:
for subset, df in [("Reference", df_test), ("Production", df_prod)]:
    prediction = predict_jleague(df)
    print(f"Subset: {subset} ({df.date.min()} to {df.date.max()})")
    for risk in ["taker", "neutral", "averse", "default"]:
        pnl, rob = get_rob(prediction, df["hcap_res"], risk)
        print(f"Risk {risk:<8}: {pnl:>6,.0f}  ({rob:>5.1%})")
    print()